# Uvicorn, Workers, and App Lifecycle

This notebook covers:

1. Running an app with Uvicorn from the CLI vs from code, and when each fits
2. Scaling with `--workers` (and Gunicorn on Linux) — what changes when there are N processes
3. Wiring startup and shutdown with the `lifespan` context manager, exercised both by `TestClient` and a real Uvicorn process
4. `--reload` vs production: what each is for, and what they're incompatible with
5. Configuring Uvicorn's own logging without fighting the framework loggers

**Scope**: FastAPI + Uvicorn. Unlike the rest of the curriculum, this notebook **launches real Uvicorn processes** via `subprocess` and hits them with `httpx` over `127.0.0.1`. Every server is terminated at the end of its cell; nothing is left running.

## 1. The `uvicorn` CLI

Uvicorn is an ASGI server. From the CLI it takes an app reference like `module:attribute` and serves it over HTTP/1.1:

```
uvicorn tiny_app:app --host 127.0.0.1 --port 8000 --log-level info
```

The `tiny_app:app` part is **importable** — Uvicorn's worker process will `import tiny_app` and look up the `app` attribute. That importability is the constraint that makes `--workers` work later: each worker has to be able to recreate the same app from scratch.

For everything below we'll:

1. Write a small app to a temp directory.
2. Spawn `python -m uvicorn ... --port <free port>` as a subprocess.
3. Wait until the port answers a request.
4. Hit it with `httpx`.
5. Shut it down.

A helper does steps 2–3 and the symmetric stop; the rest of the notebook uses it.

In [1]:
import contextlib
import os
import socket
import subprocess
import sys
import time
from pathlib import Path
from tempfile import mkdtemp

import httpx

APP_DIR = Path(mkdtemp(prefix="uvicorn_ch8_"))
print("workspace:", APP_DIR)

def free_port() -> int:
    """Ask the OS for an unused TCP port. There is a tiny race between
    closing this socket and uvicorn binding it — fine for a tutorial."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def stop_proc(proc: subprocess.Popen) -> None:
    """Kill a uvicorn process *and its children* on Windows.

    Plain `proc.terminate()` only kills the master; --workers children
    are left orphaned. `taskkill /T /F` walks the process tree."""
    if proc.poll() is not None:
        return
    if sys.platform == "win32":
        subprocess.run(
            ["taskkill", "/T", "/F", "/PID", str(proc.pid)],
            capture_output=True, check=False,
        )
    else:
        proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()

def launch_uvicorn(cli_args: list[str], port: int, *, cwd: Path = APP_DIR) -> subprocess.Popen:
    """Spawn `python -m uvicorn ...` and wait until it answers on `port`."""
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", *cli_args],
        cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    deadline = time.monotonic() + 10.0
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            out = (proc.stdout.read() or b"") + (proc.stderr.read() or b"")
            raise RuntimeError(f"uvicorn died at startup:\n{out.decode(errors='replace')}")
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                return proc
        except OSError:
            time.sleep(0.1)
    stop_proc(proc)
    raise TimeoutError(f"uvicorn did not bind 127.0.0.1:{port} in time")

# Write the tiniest possible FastAPI app to APP_DIR.
(APP_DIR / "tiny_app.py").write_text("""from fastapi import FastAPI
app = FastAPI()

@app.get("/")
def root():
    return {"hello": "uvicorn"}
""", encoding="utf-8")

print("tiny_app.py written")


workspace: C:\Users\mathi\AppData\Local\Temp\uvicorn_ch8_kkc3n57b
tiny_app.py written


Now we run it. The cell below launches `python -m uvicorn tiny_app:app` on a free port, fires a request from `httpx`, and tears the server down. Pay attention to the response object — it's a real HTTP round-trip, not the `TestClient` ASGI shortcut we used everywhere in chapters 1–7.

In [2]:
port = free_port()
proc = launch_uvicorn(
    ["tiny_app:app", "--host", "127.0.0.1", "--port", str(port), "--log-level", "warning"],
    port,
)
try:
    r = httpx.get(f"http://127.0.0.1:{port}/", timeout=5.0)
    print("status :", r.status_code)
    print("body   :", r.json())
    print("server :", r.headers.get("server"))
finally:
    stop_proc(proc)
print("server stopped")


status : 200
body   : {'hello': 'uvicorn'}
server : uvicorn
server stopped


Two details worth noticing:

- The `server` header reports `uvicorn` — that's the wire telling you a real server answered, not an in-process `TestClient` shim.
- We passed `--log-level warning` so the access log doesn't drown the cell. In production you usually want `info`; in CI you might want `warning` plus structured access logs (see section 6).

A quick mental map of the CLI flags you'll actually use:

| Flag | What it controls | Notes |
| --- | --- | --- |
| `--host`, `--port` | Bind address. | Use `0.0.0.0` in containers, never on a laptop. |
| `--log-level` | Uvicorn's own logger verbosity. | `critical / error / warning / info / debug / trace`. |
| `--no-access-log` | Drops the per-request access log line. | Useful when you have proper request logs in app middleware. |
| `--workers N` | Process count. | Section 3. Incompatible with `--reload`. |
| `--reload` | Watch files, restart on change. | Dev only. Section 5. |
| `--proxy-headers` + `--forwarded-allow-ips` | Trust `X-Forwarded-For` from a TLS terminator. | Required behind nginx / a cloud LB. |
| `--lifespan on/off/auto` | Whether to run the ASGI lifespan protocol. | Leave on `auto`; turn off only for ASGI apps that don't implement it. |

## 2. Running Uvicorn Programmatically

Sometimes you want the server in code — embedded tooling, an integration harness that needs precise control over startup, a Click CLI that wraps Uvicorn with its own flags. Two equivalent shapes:

```python
# (a) the blocking convenience function
import uvicorn
uvicorn.run("tiny_app:app", host="127.0.0.1", port=8000, log_level="info")

# (b) the explicit two-step form — needed if you want to inspect Config
config = uvicorn.Config("tiny_app:app", host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
server.run()
```

Both block until the process is signalled. There is no useful "background mode" baked in — if you need that, run the server in a subprocess (which is what this notebook does) or in a dedicated asyncio task. The cell below uses shape (a) inside its own Python file, then we launch *that* file as a subprocess.

In [3]:
(APP_DIR / "programmatic_app.py").write_text("""import sys
import uvicorn
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"mode": "programmatic", "py": sys.version_info[:2]}

if __name__ == "__main__":
    # The port comes from argv so we can pick a free one from the notebook.
    uvicorn.run(app, host="127.0.0.1", port=int(sys.argv[1]), log_level="warning")
""", encoding="utf-8")

port = free_port()
proc = subprocess.Popen(
    [sys.executable, "programmatic_app.py", str(port)],
    cwd=APP_DIR, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
# Same readiness wait as launch_uvicorn — copied inline so the example is self-contained.
deadline = time.monotonic() + 10.0
while time.monotonic() < deadline:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.2):
            break
    except OSError:
        time.sleep(0.1)

try:
    r = httpx.get(f"http://127.0.0.1:{port}/", timeout=5.0)
    print(r.status_code, r.json())
finally:
    stop_proc(proc)


200 {'mode': 'programmatic', 'py': [3, 14]}


When to reach for each:

- **CLI** is the default. Twelve-factor deployments, Dockerfile `CMD`, systemd units, Kubernetes `command:` — every one of them ends up invoking `uvicorn` with flags.
- **Programmatic** is for *tools that contain a server*: a `tools/dev.py` that wraps Uvicorn with sane local defaults; an integration test runner that needs to start a real server in another process; a CLI you ship to users that has to expose only the flags you support.
- The thing **neither** shape gives you cleanly is starting Uvicorn in the *same* process as the cell that started it. For that you'd use `Config + Server.serve()` inside an asyncio task — at which point you're back to coordinating shutdown via signals, and a subprocess is usually clearer.

## 3. `--workers` and Gunicorn

A single Uvicorn process runs a single Python interpreter, which means a single GIL. Async I/O concurrency scales beautifully inside that process; CPU work does not. The classic remedy is *N processes, each running the same ASGI app*:

```
uvicorn worker_app:app --workers 4 --host 0.0.0.0 --port 8000
```

What that does:

- A **master** process binds the listening socket.
- It `fork()` / spawns **N worker** processes that inherit the socket and `accept()` from it.
- The kernel load-balances incoming connections across workers.
- Each worker is its own Python interpreter — separate GIL, separate in-memory state, separate cache.

The last point is the one that bites you in practice: a per-process Python dict cache no longer behaves as a global cache; a `Counter()` you increment in one request will not have moved in the next request if it landed on a different worker. The fix is always the same: move shared state into a process external to the workers (Redis, a database, a real cache).

Below we verify all this by writing a `/whoami` endpoint that returns the worker's PID, then firing 40 requests and counting distinct PIDs.

In [4]:
(APP_DIR / "worker_app.py").write_text("""import os
from fastapi import FastAPI

app = FastAPI()

@app.get("/whoami")
def whoami():
    return {"pid": os.getpid()}
""", encoding="utf-8")

port = free_port()
proc = launch_uvicorn(
    ["worker_app:app", "--host", "127.0.0.1", "--port", str(port),
     "--workers", "2", "--log-level", "warning"],
    port,
)
try:
    # Give the master a moment to spawn both workers before we start probing.
    time.sleep(1.5)
    # IMPORTANT: open a fresh httpx call per request. A reused `httpx.Client`
    # keeps HTTP/1.1 keep-alive on a single TCP connection, which on Windows
    # stays pinned to whichever worker accepted that connection — you'd see
    # only one PID. A new connection per call lets the kernel rotate accept()
    # across the workers.
    pids: list[int] = []
    for _ in range(40):
        pids.append(httpx.get(f"http://127.0.0.1:{port}/whoami", timeout=5.0).json()["pid"])
    unique = sorted(set(pids))
    print("unique PIDs seen :", unique)
    print("count            :", len(unique))
    print("our PID          :", os.getpid(), "(notebook process — not in the set)")
finally:
    stop_proc(proc)


unique PIDs seen : [16644, 20776]
count            : 2
our PID          : 9520 (notebook process — not in the set)


Two unique PIDs, neither equal to the notebook's own — that's the master spawning two workers and the kernel distributing requests between them. Try bumping `--workers` to 4 and you'll see four PIDs.

A few practical truths the experiment quietly demonstrates:

- **No shared memory.** Each worker has its own `worker_app` module, its own `app` object, its own everything. State has to be externalized.
- **No deterministic routing.** The kernel decides which worker `accept()`s a given connection. Sticky routing (e.g., one user → one worker) is not a feature of `--workers`; it's a feature of a load balancer above multiple uvicorns.
- **Connection reuse hides the multi-worker behaviour.** HTTP/1.1 keep-alive over a single `httpx.Client` pins to the worker that accepted that TCP connection — on Windows you'd see only one PID even with `--workers 4`. The fresh-`httpx.get` per request above is what lets the kernel rotate `accept()` across workers and exposes them all. The production reality is closer to the keep-alive case: clients with connection pools, sticky to whichever worker they handshook with until the pool churns.
- **Windows note.** Unix uvicorn forks, which is fast and shares pre-loaded modules with copy-on-write semantics. Windows has no `fork()`, so it `spawn()`s — each worker re-imports your app from scratch. Startup is slower; large eager imports cost N× rather than 1×.

### Gunicorn on Linux

On Linux you'll often see Gunicorn fronting Uvicorn:

```
gunicorn -w 4 -k uvicorn.workers.UvicornWorker worker_app:app
```

Gunicorn is the master; Uvicorn is the per-worker server. You'd reach for this combo when you want Gunicorn's mature worker supervision — health checks, graceful reloads via `SIGHUP`, configurable timeouts. Modern Uvicorn has caught up on most of that and is fine standalone for the majority of services. Gunicorn does not run on Windows, which is why we demoed `uvicorn --workers` directly.

## 4. The `lifespan` Context Manager

Real apps have things to set up and tear down: open a database pool, warm a cache, register a Prometheus collector, close them all on shutdown. ASGI defines a **lifespan protocol** that fires before the first request and after the last; FastAPI exposes it as a `@asynccontextmanager` you hand to `FastAPI(lifespan=...)`.

The shape:

```python
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app):
    # startup work
    yield
    # shutdown work
```

Pre-yield is startup; post-yield is shutdown. Anything you stash on `app.state` is reachable from request handlers via `request.app.state.<name>`. The old `@app.on_event("startup")` / `@app.on_event("shutdown")` decorators still work but are deprecated in favour of this single context manager — one place, predictable ordering, real exception propagation.

In [5]:
import contextlib as _ctx
import io as _io
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

class FakeDb:
    """Stand-in for a real connection pool — open() and close() print so
    you can see when the lifespan fires."""
    def __init__(self) -> None:
        self.opened = False

    def open(self) -> None:
        self.opened = True
        print("[FakeDb] connection pool opened")

    def close(self) -> None:
        self.opened = False
        print("[FakeDb] connection pool closed")

    def ping(self) -> str:
        return "pong" if self.opened else "closed"

@asynccontextmanager
async def lifespan(app: FastAPI):
    db = FakeDb()
    db.open()
    app.state.db = db
    yield
    db.close()

app = FastAPI(lifespan=lifespan)

@app.get("/ping")
def ping(request: Request):
    return {"db": request.app.state.db.ping()}

# Use TestClient as a context manager so the lifespan actually fires.
buf = _io.StringIO()
with _ctx.redirect_stdout(buf):
    with TestClient(app) as client:
        r1 = client.get("/ping")
        r2 = client.get("/ping")
    # leaving the `with` block triggers lifespan shutdown
print("captured lifespan logs:")
print(buf.getvalue())
print("responses:", r1.json(), r2.json())


captured lifespan logs:
[FakeDb] connection pool opened
[FakeDb] connection pool closed

responses: {'db': 'pong'} {'db': 'pong'}


C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


The `with TestClient(app) as client:` block is what enables the lifespan to run. If you write `client = TestClient(app)` without `with`, `TestClient` enters and exits the lifespan **around each individual call** — which is fine for one-off tests but masks ordering bugs you'd catch in production. Always use the context-manager form when the lifespan does real work.

Now let's prove the same `lifespan` runs identically under a real Uvicorn process. We'll write the same app to a file, launch it, and capture Uvicorn's stderr to see the startup print.

In [6]:
(APP_DIR / "lifespan_app.py").write_text("""import sys
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request

class FakeDb:
    def __init__(self):
        self.opened = False
    def open(self):
        self.opened = True
        print("[FakeDb] connection pool opened", flush=True)
    def close(self):
        self.opened = False
        print("[FakeDb] connection pool closed", flush=True)
    def ping(self):
        return "pong" if self.opened else "closed"

@asynccontextmanager
async def lifespan(app):
    db = FakeDb()
    db.open()
    app.state.db = db
    yield
    db.close()

app = FastAPI(lifespan=lifespan)

@app.get("/ping")
def ping(request: Request):
    return {"db": request.app.state.db.ping(), "py": sys.version_info[:2]}
""", encoding="utf-8")

port = free_port()
proc = launch_uvicorn(
    ["lifespan_app:app", "--host", "127.0.0.1", "--port", str(port), "--log-level", "info"],
    port,
)
try:
    r = httpx.get(f"http://127.0.0.1:{port}/ping", timeout=5.0)
    print("response :", r.json())
finally:
    stop_proc(proc)

# stop_proc terminated the process; its captured stdout now contains
# both the startup and shutdown prints.
out = (proc.stdout.read() or b"").decode(errors="replace")
err = (proc.stderr.read() or b"").decode(errors="replace")
print("---- subprocess stdout ----")
print(out.strip() or "(empty)")
print("---- subprocess stderr (uvicorn logs) ----")
# Trim the long startup banner; just show the lines that prove the lifecycle.
for line in err.splitlines():
    if any(k in line for k in ("Started", "Waiting", "Application startup",
                                "Application shutdown", "Shutting down")):
        print(line)


response : {'db': 'pong', 'py': [3, 14]}
---- subprocess stdout ----
[FakeDb] connection pool opened
INFO:     127.0.0.1:60835 - "GET /ping HTTP/1.1" 200 OK
---- subprocess stderr (uvicorn logs) ----
INFO:     Started server process [21864]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


The stdout block shows `[FakeDb] connection pool opened` *before* the request was served, and `[FakeDb] connection pool closed` after we sent the kill signal — exactly the same ordering `TestClient` exposed. That symmetry is the point: your lifespan is testable in-process; if it works under `TestClient` it works under Uvicorn.

A subtlety worth keeping in mind: with `--workers N`, lifespan runs **once per worker**. If startup opens a connection pool with 10 connections and you scale to 4 workers, you've actually opened 40 connections. Pool sizes are per-process. Same rule when you size DB pools, thread pools, anything else stateful.

## 5. `--reload` vs Production

`--reload` watches your source tree and restarts the server when a file changes. It is the right default for local development and the wrong default for everything else.

What it really does:

- Starts a **single** process. There is no master/worker split; the file watcher *is* the server.
- Watches `.py` (and optional `--reload-include` patterns) under the working directory.
- On change, it kills the running process and starts a new one. In-flight requests are dropped.

What it cannot do:

- **Coexist with `--workers`.** Uvicorn refuses the combination. The reload model assumes one process; workers assume a master. Pick one.
- **Survive partial saves.** Editors that write files in two steps (truncate + write) can briefly produce a syntax-error state. Reload sees both saves and may restart on an unparseable file — you'll see a traceback and an exit. Re-save fixes it.
- **Match production behaviour.** Different process model, different startup ordering, no graceful shutdown. The further your dev loop drifts from prod, the more you'll be debugging the gap.

The production stance is:

```
uvicorn app:app --host 0.0.0.0 --port 8000 --workers $(nproc) --proxy-headers
```

…run under a supervisor (systemd, Docker, Kubernetes) that owns process lifetime. The supervisor sends `SIGTERM`; Uvicorn drains in-flight requests up to `--timeout-graceful-shutdown` and then exits. That's the contract a load balancer is built around — `SIGTERM` → 200s until the deadline → 503s after — and `--reload` doesn't play in that world at all.

## 6. Logging Configuration

Uvicorn ships its own logging config — three named loggers:

- `uvicorn` — server-level events (lifecycle, errors).
- `uvicorn.error` — request handler exceptions.
- `uvicorn.access` — the access log line per request.

You override any of them by passing a `log_config` dict (or a path to a YAML file with `--log-config`). The shape is `logging.config.dictConfig` — formatters, handlers, loggers. The single most common customization is moving access logs to JSON so an aggregator (Loki, CloudWatch, Datadog) can index them.

In [7]:
(APP_DIR / "json_log_app.py").write_text("""import sys
import uvicorn
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"ok": True}

LOG_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "json": {
            # Minimal, dependency-free JSON-ish line. In real life you'd reach
            # for python-json-logger or structlog (see notebook 6.2).
            "format": '{\"level\":\"%(levelname)s\",\"logger\":\"%(name)s\",\"msg\":%(message)r}',
        },
    },
    "handlers": {
        "stderr": {"class": "logging.StreamHandler", "stream": "ext://sys.stderr", "formatter": "json"},
    },
    "loggers": {
        "uvicorn":        {"handlers": ["stderr"], "level": "INFO", "propagate": False},
        "uvicorn.error":  {"handlers": ["stderr"], "level": "INFO", "propagate": False},
        "uvicorn.access": {"handlers": ["stderr"], "level": "INFO", "propagate": False},
    },
}

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=int(sys.argv[1]), log_config=LOG_CONFIG)
""", encoding="utf-8")

port = free_port()
proc = subprocess.Popen(
    [sys.executable, "json_log_app.py", str(port)],
    cwd=APP_DIR, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
deadline = time.monotonic() + 10.0
while time.monotonic() < deadline:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=0.2):
            break
    except OSError:
        time.sleep(0.1)

try:
    httpx.get(f"http://127.0.0.1:{port}/", timeout=5.0)
finally:
    stop_proc(proc)

err = (proc.stderr.read() or b"").decode(errors="replace")
print("---- uvicorn stderr (JSON-shaped lines) ----")
for line in err.splitlines():
    if line.strip().startswith("{"):
        print(line)


---- uvicorn stderr (JSON-shaped lines) ----
{"level":"INFO","logger":"uvicorn.error","msg":'Started server process [21948]'}
{"level":"INFO","logger":"uvicorn.error","msg":'Waiting for application startup.'}
{"level":"INFO","logger":"uvicorn.error","msg":'Application startup complete.'}
{"level":"INFO","logger":"uvicorn.error","msg":'Uvicorn running on http://127.0.0.1:60836 (Press CTRL+C to quit)'}
{"level":"INFO","logger":"uvicorn.access","msg":'127.0.0.1:60845 - "GET / HTTP/1.1" 200'}


Three JSON-ish lines for the lifecycle and one for the access log. In production you'd use `structlog`'s `JSONRenderer` (notebook 6.2) so app logs and server logs land in the same format, and you'd add a `correlation_id` filter so request IDs propagate.

One trap to remember: if `disable_existing_loggers` is `True` (the dictConfig default — Uvicorn sets it to `False`), every logger created *before* `dictConfig` runs gets silently disabled. That includes loggers FastAPI uses internally. Keep `False` unless you have a very specific reason to flip it.

## Key Takeaways

- **Uvicorn is the runtime.** From the CLI in production, from `uvicorn.run` for embedded tooling, from a subprocess for integration tests like the ones in this notebook.
- **`--workers N` is N processes, not N threads.** No shared memory, no in-process global cache, no deterministic routing. State that has to be shared belongs outside the workers.
- **`lifespan` is the one true place** to open and close per-process resources. It runs once per worker; size pools accordingly. `TestClient(app)` as a context manager exercises the same hook your prod server will.
- **`--reload` is dev-only.** Single process, no graceful shutdown, incompatible with `--workers`. Production runs N workers under a supervisor that owns the `SIGTERM` contract.
- **Uvicorn's logging is a normal `dictConfig`.** Override `uvicorn`, `uvicorn.error`, `uvicorn.access`; in prod, render JSON and let your aggregator index it.
- **Capstone tie-in**: the `lifespan` defined here — open a `Settings`-driven repository on startup, close it on shutdown, stash on `app.state` — is exactly the shape `examples/portfolio_analytics_api/src/portfolio_api/main.py` will use.

## Exercises

**1. Two endpoints, one process.** Extend `tiny_app.py` with a `/healthz` route that returns `{"status": "ok"}` and a `/readyz` route that returns `{"status": "ready"}` only after a small warmup delay (use the lifespan to set `app.state.ready = True` after `await asyncio.sleep(1.0)`). Launch under Uvicorn from this notebook, hit `/readyz` immediately *and* after a 1.5s wait, and assert the status flips. The point: liveness vs readiness, exposed by the lifespan.

**2. Measure the workers-vs-single throughput.** Reuse `worker_app.py` and add a CPU-bound endpoint `/fib` that computes `fib(30)` recursively. Run the server with `--workers 1` and time how long 20 concurrent `/fib` calls take with `httpx.AsyncClient`. Then re-run with `--workers 4` and compare. The ratio should be close to the worker count for pure CPU work; observe what happens for a pure-I/O endpoint like `/whoami`.

**3. Shared state breaks with workers.** Add an in-process counter to `worker_app.py` — a module-level `COUNTS: dict[int, int]` keyed by PID, incremented in a `/count` endpoint. Run with `--workers 2`, fire 50 requests, and read the counts. Show that no worker's counter reaches 50, then sketch the one-line change you'd make to fix it (you don't have to implement Redis — just describe the mechanism). The point: this is the production gotcha workers exist to surface.